# Нейросеть на Python шаг за шагом

В этом ноутбуке мы повторим схему с картинки:

- **2 входа**: `x1`, `x2`
- **1 скрытый слой** из **3 нейронов**
- **2 скрытый слой** из **2 нейронов**
- **1 выход**: `y`

Мы сделаем это **пошагово** и максимально просто, без `tensorflow` и `pytorch`, чтобы понять **математику и логику**.

## Шаг 1. Импортируем библиотеки

Нам нужны:

- `numpy` — для матриц и вычислений
- `matplotlib` — чтобы посмотреть данные

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Шаг 2. Создадим маленький датасет

Сделаем простую задачу классификации **XOR**.

Правила такие:

- `(0, 0) -> 0`
- `(0, 1) -> 1`
- `(1, 0) -> 1`
- `(1, 1) -> 0`

Эта задача хороша тем, что **одного слоя недостаточно**.  
Именно поэтому тут полезна **многослойная нейросеть**.

In [ ]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y = np.array([
    [0],
    [1],
    [1],
    [0]
], dtype=float)

print("X shape:", X.shape)
print("y shape:", y.shape)
print()
print("X =")
print(X)
print()
print("y =")
print(y)

## Шаг 3. Посмотрим на данные

По оси X будет `x1`, по оси Y будет `x2`.

In [ ]:
for i in range(len(X)):
    label = int(y[i, 0])
    plt.scatter(X[i, 0], X[i, 1], s=120)
    plt.text(X[i, 0] + 0.02, X[i, 1] + 0.02, f"class={label}", fontsize=11)

plt.xlim(-0.2, 1.2)
plt.ylim(-0.2, 1.2)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Датасет XOR")
plt.grid(True)
plt.show()

## Шаг 4. Вспомним идею одного нейрона

Один нейрон считает:

\[
z = x_1 w_1 + x_2 w_2 + b
\]

потом применяет функцию активации:

\[
a = f(z)
\]

Где:

- `w` — веса
- `b` — bias (смещение)
- `f` — функция активации

## Шаг 5. Выберем функцию активации

Возьмём **sigmoid**:

\[
\sigma(z) = \frac{1}{1 + e^{-z}}
\]

Она переводит число в диапазон от **0 до 1**.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    # a = sigmoid(z)
    return a * (1 - a)

In [ ]:
test_values = np.array([-3, -1, 0, 1, 3], dtype=float)
print("test values:", test_values)
print("sigmoid:", sigmoid(test_values))

## Шаг 6. Опишем архитектуру сети

Наша сеть будет такая:

- вход: **2**
- скрытый слой 1: **3**
- скрытый слой 2: **2**
- выход: **1**

То есть:

`2 -> 3 -> 2 -> 1`

## Шаг 7. Инициализируем веса

Матрицы весов:

- `W1`: из 2 входов в 3 нейрона  → размер `(2, 3)`
- `W2`: из 3 нейронов в 2 нейрона → размер `(3, 2)`
- `W3`: из 2 нейронов в 1 выход   → размер `(2, 1)`

Bias:

- `b1`: `(1, 3)`
- `b2`: `(1, 2)`
- `b3`: `(1, 1)`

In [ ]:
np.random.seed(42)

W1 = np.random.randn(2, 3) * 0.5
b1 = np.zeros((1, 3))

W2 = np.random.randn(3, 2) * 0.5
b2 = np.zeros((1, 2))

W3 = np.random.randn(2, 1) * 0.5
b3 = np.zeros((1, 1))

print("W1 shape:", W1.shape)
print("b1 shape:", b1.shape)
print("W2 shape:", W2.shape)
print("b2 shape:", b2.shape)
print("W3 shape:", W3.shape)
print("b3 shape:", b3.shape)

## Шаг 8. Forward propagation — прямой проход

Считаем по слоям:

### Первый скрытый слой
\[
z_1 = XW_1 + b_1
\]
\[
a_1 = \sigma(z_1)
\]

### Второй скрытый слой
\[
z_2 = a_1W_2 + b_2
\]
\[
a_2 = \sigma(z_2)
\]

### Выход
\[
z_3 = a_2W_3 + b_3
\]
\[
\hat{y} = a_3 = \sigma(z_3)
\]

In [ ]:
def forward(X, W1, b1, W2, b2, W3, b3):
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)

    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)

    z3 = a2 @ W3 + b3
    a3 = sigmoid(z3)

    cache = {
        "z1": z1, "a1": a1,
        "z2": z2, "a2": a2,
        "z3": z3, "a3": a3
    }
    return a3, cache

In [ ]:
y_pred, cache = forward(X, W1, b1, W2, b2, W3, b3)

print("Начальные предсказания:")
print(y_pred)

## Шаг 9. Функция ошибки

Возьмём **MSE**:

\[
L = \frac{1}{n} \sum (y - \hat{y})^2
\]

In [ ]:
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

loss = mse_loss(y, y_pred)
print("Начальная ошибка:", loss)

## Шаг 10. Идея backpropagation

Теперь нужно понять:

**как менять веса, чтобы ошибка уменьшалась**.

Для этого считаем градиенты **с конца к началу**.

Схема:

1. считаем ошибку на выходе
2. переносим её назад во второй скрытый слой
3. переносим её назад в первый скрытый слой
4. обновляем веса

## Шаг 11. Backpropagation

Ниже формулы уже записаны в коде.

Главная идея:

- сначала находим, насколько ошибся выход
- потом понимаем, какой вклад дали предыдущие слои
- потом двигаем веса в сторону уменьшения ошибки

In [ ]:
def backward(X, y, W1, b1, W2, b2, W3, b3, cache):
    a1 = cache["a1"]
    a2 = cache["a2"]
    a3 = cache["a3"]

    m = X.shape[0]

    # dL/da3 для MSE
    da3 = (2 / m) * (a3 - y)
    dz3 = da3 * sigmoid_derivative(a3)

    dW3 = a2.T @ dz3
    db3 = np.sum(dz3, axis=0, keepdims=True)

    da2 = dz3 @ W3.T
    dz2 = da2 * sigmoid_derivative(a2)

    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    da1 = dz2 @ W2.T
    dz1 = da1 * sigmoid_derivative(a1)

    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    grads = {
        "dW1": dW1, "db1": db1,
        "dW2": dW2, "db2": db2,
        "dW3": dW3, "db3": db3
    }
    return grads

In [ ]:
grads = backward(X, y, W1, b1, W2, b2, W3, b3, cache)

for name, value in grads.items():
    print(name)
    print(value)
    print()

## Шаг 12. Обновление весов

Формула простая:

\[
W = W - \eta \cdot \frac{\partial L}{\partial W}
\]

где:

- `η` — learning rate

In [ ]:
def update_params(W1, b1, W2, b2, W3, b3, grads, lr=1.0):
    W1 = W1 - lr * grads["dW1"]
    b1 = b1 - lr * grads["db1"]

    W2 = W2 - lr * grads["dW2"]
    b2 = b2 - lr * grads["db2"]

    W3 = W3 - lr * grads["dW3"]
    b3 = b3 - lr * grads["db3"]

    return W1, b1, W2, b2, W3, b3

## Шаг 13. Соберём обучение в цикл

Теперь повторяем:

1. forward
2. loss
3. backward
4. update

много раз.

In [ ]:
np.random.seed(42)

W1 = np.random.randn(2, 3) * 0.5
b1 = np.zeros((1, 3))

W2 = np.random.randn(3, 2) * 0.5
b2 = np.zeros((1, 2))

W3 = np.random.randn(2, 1) * 0.5
b3 = np.zeros((1, 1))

lr = 1.0
epochs = 10000
loss_history = []

for epoch in range(epochs):
    y_pred, cache = forward(X, W1, b1, W2, b2, W3, b3)
    loss = mse_loss(y, y_pred)
    loss_history.append(loss)

    grads = backward(X, y, W1, b1, W2, b2, W3, b3, cache)
    W1, b1, W2, b2, W3, b3 = update_params(W1, b1, W2, b2, W3, b3, grads, lr=lr)

    if epoch % 1000 == 0:
        print(f"epoch={epoch}, loss={loss:.6f}")

print(f"epoch={epochs}, loss={loss_history[-1]:.6f}")

## Шаг 14. Посмотрим, как уменьшалась ошибка

In [ ]:
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Изменение ошибки во время обучения")
plt.grid(True)
plt.show()

## Шаг 15. Проверим предсказания после обучения

In [ ]:
y_pred, cache = forward(X, W1, b1, W2, b2, W3, b3)

print("Сырые предсказания:")
print(y_pred)

print("\nОкругленные предсказания:")
print((y_pred > 0.5).astype(int))

print("\nПравильные ответы:")
print(y.astype(int))

## Шаг 16. Сделаем удобную функцию predict

In [ ]:
def predict(X_new, W1, b1, W2, b2, W3, b3):
    y_pred, _ = forward(X_new, W1, b1, W2, b2, W3, b3)
    return (y_pred > 0.5).astype(int), y_pred

classes, probs = predict(X, W1, b1, W2, b2, W3, b3)

print("classes:")
print(classes)

print("\nprobabilities:")
print(probs)

## Шаг 17. Что тут важно понять

### 1. Нейросеть — это цепочка матриц
На каждом слое происходит:

- линейное преобразование
- потом активация

### 2. Слои учатся весам
Именно веса `W1`, `W2`, `W3` делают сеть умной.

### 3. Backpropagation
Нужен для ответа на вопрос:

**какие именно веса надо поменять и на сколько**

### 4. Gradient descent
Это уже сам шаг обновления:

\[
W := W - lr \cdot grad
\]

## Шаг 18. Как это связано с рисунком

На картинке было примерно так:

- входы: `x1`, `x2`
- скрытый слой из 3 нейронов
- скрытый слой из 2 нейронов
- выход `y`

В этом ноутбуке мы сделали **ровно такую же идею** на Python.

## Шаг 19. Что можно попробовать дальше

1. Поменять количество нейронов  
2. Поменять `sigmoid` на `tanh` или `ReLU`  
3. Увеличить количество точек  
4. Сделать классификацию не XOR, а другой задачи  
5. Сравнить обучение при разных `learning rate`

## Шаг 20. Мини-итог

Ты сейчас сделал базовый **MLP с нуля**:

- forward pass
- loss
- backward pass
- обновление весов
- предсказание

Это уже фундамент нейросетей.